In [1]:
import os
import json
import torch
import cv2

CLASS_MAP = {
    "speedlimit": 1,
    "crosswalk": 2,
    "trafficlight": 3,
    "stop": 4
}

class NinjaDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, ann_dir):
        self.img_dir = img_dir
        self.ann_dir = ann_dir

        self.images = sorted([
            f for f in os.listdir(img_dir)
            if f.endswith(".png")
        ])

    def __getitem__(self, idx):
        img_name = self.images[idx]

        img_path = os.path.join(self.img_dir, img_name)
        ann_path = os.path.join(self.ann_dir, img_name + ".json")  
        # <-- handles road0.png → road0.png.json

        # ==== Load image ====
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        boxes = []
        labels = []

        # ==== Load annotation ====
        if os.path.exists(ann_path):
            with open(ann_path) as f:
                data = json.load(f)

            for obj in data.get("objects", []):
                x1, y1 = obj["points"]["exterior"][0]
                x2, y2 = obj["points"]["exterior"][1]

                # clamp to image size
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)

                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(CLASS_MAP[obj["classTitle"]])

        # ==== Handle empty case ====
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }

        img = torch.tensor(img / 255.0, dtype=torch.float32).permute(2, 0, 1)

        return img, target

    def __len__(self):
        return len(self.images)

In [2]:
from torch.utils.data import DataLoader
from torch.utils.data import random_split

def collate_fn(batch):
    return tuple(zip(*batch))

dataset = NinjaDataset(
    img_dir=r"D:\617 project\raw_data\road-sign-detection-DatasetNinja\ds\img",
    ann_dir=r"D:\617 project\raw_data\road-sign-detection-DatasetNinja\ds\ann"
)

train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,       
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=True
)

In [3]:
import torchvision

def get_model(num_classes=5, freeze_backbone=False):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights="DEFAULT"
    )


    in_features = model.roi_heads.box_predictor.cls_score.in_features

    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
        in_features,
        num_classes 
    )


    if freeze_backbone:
        for param in model.backbone.parameters():
            param.requires_grad = False

    return model

In [10]:
import torch
import wandb


def train_model(train_loader, val_loader, epochs=20):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ==== Model ====
    model = get_model(num_classes=5, freeze_backbone=False)
    model.to(device)

    # ==== Optimizer ====
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

    # ==== W&B ====
    wandb.init(
    project="faster-rcnn-traffic",
    name="run1",
   
)

    # ==== Training ====
    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for images, targets in train_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}")
        wandb.log({"train/loss": train_loss})

        # ==== Validation ====
        model.eval()
        val_loss = 0

        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

                loss_dict = model(images, targets)
                loss = sum(loss_dict.values())
                val_loss += loss.item()

        print(f"Epoch {epoch}: Val Loss = {val_loss:.4f}")
        wandb.log({"val/loss": val_loss})

    # ==== Save model ====
    torch.save(model.state_dict(), "faster_rcnn_transfer.pth")
    wandb.finish()

    return model

In [11]:
import wandb
import os
os.environ["WANDB_START_METHOD"] = "thread"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
wandb.login()

True

In [12]:
model = train_model(train_loader, val_loader, epochs=20)

ServicePollForTokenError: Failed to read port info after 30.0 seconds.

In [13]:
model = get_model(num_classes=5, freeze_backbone=True)

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to C:\Users\john2/.cache\torch\hub\checkpoints\fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:03<00:00, 42.5MB/s] 
